# 03 · DPO v2, with the reference model fixed

**Decision this notebook supports:** does preference training on the model's own mistakes reduce them, without moving the model away from what SFT v2 learned?

**The v1 bug fixed here.** v1 kept training the SFT adapter in place and passed no reference model. With PEFT, TRL then uses "the model with its adapter switched off" as the reference, which was **plain base Qwen, not SFT**. DPO's KL leash pointed at the wrong model.

v2 does it properly:
1. Merge the SFT v2 adapter into the base weights, giving a full "SFT v2" model on disk.
2. Train a **new** LoRA on top of that merged model.
3. The reference is now "merged model, new adapter off" = **SFT v2 exactly**. A cell below proves it numerically.

Prompts are in the serving format (ENGINEERED + documents + question), not v1's short `SYSTEM` prompt.

In [ ]:
# Find the v2 folder, put it (and the main lab one folder up) on the import path,
# and apply the memory-safety settings BEFORE torch is imported.
import sys, json, time, re, math, random, hashlib, collections
from pathlib import Path
V2_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "configs/v2.yaml").is_file())
sys.path.insert(0, str(V2_ROOT))
from v2lib import setup, device as D, prompts as P, facts as F
CFG = setup.cfg()
SNAP = setup.read_json(V2_ROOT / CFG["retrieval"]["snapshot"])
CHUNKS = SNAP["chunks"]
print("v2 folder:", V2_ROOT)
print("main lab:", setup.PARENT)
print("device:", D.device(), "| dtype:", D.dtype(), "| base model:", D.base_model_path())
D.memory_report("start")

## Step 1 · merge SFT v2 into a full model

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
DEVICE, DTYPE = D.device(), D.dtype()
tok = AutoTokenizer.from_pretrained(D.base_model_path())
MERGED = V2_ROOT / CFG["dpo"]["merged_sft_model"]
base = AutoModelForCausalLM.from_pretrained(D.base_model_path(), dtype=DTYPE).to(DEVICE)
sft = PeftModel.from_pretrained(base, str(V2_ROOT / CFG["sft"]["output_adapter"])).to(DEVICE).eval()
probe = tok.apply_chat_template(P.messages("What's the Volt X's top speed?", None), tokenize=True, add_generation_prompt=True, return_dict=False)
x = torch.tensor([probe], device=DEVICE)
with torch.inference_mode():
    sft_logits = sft(input_ids=x).logits[0, -1].float().cpu()
merged = sft.merge_and_unload()
merged.save_pretrained(str(MERGED)); tok.save_pretrained(str(MERGED))
with torch.inference_mode():
    merged_logits = merged(input_ids=x).logits[0, -1].float().cpu()
print("max |logit difference| SFT adapter vs merged model:", float((sft_logits - merged_logits).abs().max()))
del sft, merged, base
D.empty_cache(); D.memory_report("merged saved")

## Step 2 · the preference data, in the serving format

In [ ]:
from datasets import Dataset
pairs_train = setup.read_jsonl(V2_ROOT / CFG["data"]["dpo_train"])
pairs_val = setup.read_jsonl(V2_ROOT / CFG["data"]["dpo_val"])
print(len(pairs_train), "train pairs |", len(pairs_val), "validation pairs")
print("train failures:", dict(collections.Counter(p["failure"] for p in pairs_train)))

def to_ds(rows):
    return Dataset.from_list([{
        "prompt": P.messages(r["question"], [CHUNKS[c] for c in r["chunk_ids"]]),
        "chosen": [{"role": "assistant", "content": r["chosen"]}],
        "rejected": [{"role": "assistant", "content": r["rejected"]}]} for r in rows])

train_ds, val_ds = to_ds(pairs_train), to_ds(pairs_val)
def full_len(r, side):
    m = P.messages(r["question"], [CHUNKS[c] for c in r["chunk_ids"]]) + [{"role": "assistant", "content": r[side]}]
    return len(tok.apply_chat_template(m, tokenize=True, return_dict=False))
longest = max(full_len(r, s) for r in pairs_train + pairs_val for s in ("chosen", "rejected"))
MAX_LEN = int(math.ceil((longest + 8) / 64) * 64)
print(f"longest prompt+response {longest} tokens -> max_length {MAX_LEN} (nothing is truncated)")

## Step 3 · new LoRA on the merged model; reference = SFT v2

`beta` is the leash: higher keeps the policy closer to SFT v2. The learning rate is 40× smaller than SFT's, because DPO only nudges probabilities that are already shaped.

In [ ]:
from trl import DPOTrainer, DPOConfig
from peft import LoraConfig
d, lc = CFG["dpo"], CFG["lora"]
policy = AutoModelForCausalLM.from_pretrained(str(MERGED), dtype=DTYPE).to(DEVICE)
dpo_args = DPOConfig(
    output_dir=str(V2_ROOT / "runs/dpo_v2_trainer"), beta=d["beta"], loss_type=d["loss_type"],
    num_train_epochs=d["epochs"], learning_rate=d["lr"], per_device_train_batch_size=d["batch_size"],
    per_device_eval_batch_size=1, gradient_accumulation_steps=d["grad_accum"], max_length=MAX_LEN,
    truncation_mode="keep_start", precompute_ref_log_probs=True, gradient_checkpointing=d["gradient_checkpointing"],
    gradient_checkpointing_kwargs={"use_reentrant": False}, logging_steps=5, eval_strategy="epoch",
    save_strategy="no", report_to=[], seed=CFG["seed"])
peft_cfg = LoraConfig(r=lc["r"], lora_alpha=lc["alpha"], lora_dropout=lc["dropout"],
                      target_modules=lc["target_modules"], task_type="CAUSAL_LM")
from transformers import TrainerCallback
class ProgressLog(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kw):
        keep = {k: round(v, 4) for k, v in (logs or {}).items() if isinstance(v, (int, float)) and ("loss" in k or "margins" in k or "accuracies" in k)}
        setup.progress("03_dpo", f"step {state.global_step}/{state.max_steps} {keep}")
trainer = DPOTrainer(model=policy, ref_model=None, args=dpo_args, train_dataset=train_ds, eval_dataset=val_ds,
                     processing_class=tok, peft_config=peft_cfg, callbacks=[ProgressLog()])
# Proof the reference is SFT v2: with the new adapter disabled, the policy equals the merged SFT model.
x = torch.tensor([probe], device=DEVICE)
check = AutoModelForCausalLM.from_pretrained(str(MERGED), dtype=DTYPE).to(DEVICE).eval()
with torch.inference_mode():
    with trainer.model.disable_adapter():
        ref_logits = trainer.model(input_ids=x).logits[0, -1].float().cpu()
    sft_v2_logits = check(input_ids=x).logits[0, -1].float().cpu()
print("reference vs merged SFT v2, max |logit difference|:", float((ref_logits - sft_v2_logits).abs().max()))
del check; D.empty_cache(); D.memory_report("trainer ready")

## Train

Watch `rewards/margins` (chosen minus rejected) and `rewards/accuracies` on the **validation** pairs. They rise only if the model's preference really moved.

In [ ]:
t0 = time.time()
result = trainer.train()
dpo_minutes = (time.time() - t0) / 60
print(result.metrics); print(f"DPO took {dpo_minutes:.1f} minutes")
D.memory_report("after DPO")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
hist = pd.DataFrame(trainer.state.log_history)
cols = [c for c in ["rewards/margins", "rewards/accuracies", "loss"] if c in hist]
fig, axes = plt.subplots(1, len(cols), figsize=(4 * len(cols), 3))
for ax, col in zip(axes, cols):
    hist.dropna(subset=[col]).plot(x="step", y=col, ax=ax, legend=False, title=col)
plt.tight_layout(); plt.show()
ev = hist[[c for c in hist.columns if c.startswith("eval_")] + ["epoch"]].dropna(how="all", subset=[c for c in hist.columns if c.startswith("eval_")])
display(ev)

## Save, then compare SFT v2 and DPO v2 on validation prompts

In [ ]:
out = V2_ROOT / d["output_adapter"]
trainer.model.save_pretrained(str(out))
setup.write_json(V2_ROOT / "runs/dpo_v2.json", {"metrics": result.metrics, "minutes": round(dpo_minutes, 1), "max_length": MAX_LEN,
    "n_train": len(train_ds), "n_val": len(val_ds), "config": d, "reference": "merged SFT v2 (new adapter disabled)",
    "log_history": trainer.state.log_history, "saved_at": time.strftime("%Y-%m-%d %H:%M:%S")})
print("saved", out.relative_to(V2_ROOT))

model = trainer.model.eval()
def generate(question, chunk_ids, use_adapter=True):
    ids = tok.apply_chat_template(P.messages(question, [CHUNKS[c] for c in chunk_ids]), tokenize=True, add_generation_prompt=True, return_dict=False)
    x = torch.tensor([ids], device=DEVICE)
    with torch.inference_mode():
        if use_adapter:
            o = model.generate(input_ids=x, attention_mask=torch.ones_like(x), max_new_tokens=160, do_sample=False, pad_token_id=tok.pad_token_id)
        else:
            with model.disable_adapter():
                o = model.generate(input_ids=x, attention_mask=torch.ones_like(x), max_new_tokens=160, do_sample=False, pad_token_id=tok.pad_token_id)
    return tok.decode(o[0, x.shape[1]:], skip_special_tokens=True).strip()

for r in pairs_val[:8]:
    print(f"[{r['failure']}] {r['question']}\n   SFT v2 : {generate(r['question'], r['chunk_ids'], False)}\n   DPO v2 : {generate(r['question'], r['chunk_ids'], True)}\n   chosen : {r['chosen']}\n")
del model, trainer, policy
D.empty_cache(); D.memory_report("released")